In [63]:
from openai import OpenAI
from typing import Type
import os
import base64
from pdf2image import convert_from_path
from dotenv import load_dotenv
from io import BytesIO
from pydantic import BaseModel, Field
import json

import Validity_Functions

In [64]:
pdf_path = "/zfs/projects/students/ltdarc-usf-intern-2025/data/Arizona_Republic_Sun__Dec_17__2000_ (15).pdf"

In [65]:
def pdf_to_b64(pdf_path: str) -> str:
    """ Converts pdf image to b64."""
    pages = convert_from_path(pdf_path, first_page=1, last_page=1, dpi=100)
    buffer = BytesIO()
    pages[0].save(buffer, format="PNG")
    img_bytes = buffer.getvalue()
    img_b64 = base64.b64encode(img_bytes).decode()

    return img_b64

In [66]:
load_dotenv("/zfs/projects/students/ltdarc-usf-intern-2025/.env")

True

In [67]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [85]:
model = "gpt-4o-mini"

In [86]:
role_prompt = "You are a metadata extraction assistant that reading the TV guides of newspapers."


In [87]:
content_prompt = """Look at this newspaper page and extract two fields:
1. The name of the newspaper
2. The publication date (do not include day of week)

Return your answer ONLY as a valid JSON object with the following keys:
  - newspaper
  - date

Example:
{
  "newspaper": "The New York Times",
  "date": "January 5, 2023"
}
"""

In [88]:
class MetaData(BaseModel):
    newspaper_name: str
    publication_date: str
    

In [89]:
def call_llm(b64: str, model: str, role_prompt: str, content_prompt: str, structured_output: Type[BaseModel]) -> BaseModel:
    """Feeds B64 image into LLM and returns structured JSON as MetaData."""
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    response = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": [{"type": "input_text", "text": role_prompt}]},
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": content_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{b64}",  # ✅ just the string!
                    },
                ],
            },
        ],
        text_format=structured_output,
    )

    return response.output_parsed
    

In [90]:
b64 = pdf_to_b64(pdf_path)
call_llm(b64, model, role_prompt, content_prompt, MetaData)

MetaData(newspaper_name='Arizona Republic', publication_date='December 17, 2000')

In [91]:
data_path='/zfs/projects/students/ltdarc-usf-intern-2025/data'

index = Validity_Functions.make_index(data_path)

In [92]:
results = dict()

for idx, row in index.iterrows():
    pdf_path = row["pdf_files"]
    b64 = pdf_to_b64(pdf_path)

    output = call_llm(b64, model, role_prompt, content_prompt, MetaData)
    results[idx] = output

    

    

In [93]:
print(results)

{0: MetaData(newspaper_name='Arizona Republic', publication_date='December 17, 2000'), 1: MetaData(newspaper_name='Austin American-Statesman', publication_date='August 3, 2014'), 2: MetaData(newspaper_name='Austin American-Statesman', publication_date='February 22, 2015'), 3: MetaData(newspaper_name='Austin American-Statesman', publication_date='January 12, 2014'), 4: MetaData(newspaper_name='Austin American-Statesman', publication_date='November 23, 2014'), 5: MetaData(newspaper_name='Chicago Tribune', publication_date='May 28, 1995'), 6: MetaData(newspaper_name='Chicago Tribune', publication_date='October 30, 1994'), 7: MetaData(newspaper_name='Daily News', publication_date='April 13, 2008'), 8: MetaData(newspaper_name='Daily News', publication_date='April 25, 2010'), 9: MetaData(newspaper_name='Daily News', publication_date='June 17, 2012'), 10: MetaData(newspaper_name='Daily News', publication_date='June 9, 2013'), 11: MetaData(newspaper_name='Daily News', publication_date='March 1